在大模型（LLM）和超大规模多模态模型（如巨型 SigLIP、GPT 级别模型）的训练中，工程师们面临的最大敌人不是算法不够聪明，而是 **GPU 显存爆炸（OOM, Out of Memory）**。

例如，一个 100 亿（10B）参数的模型，光是把模型权重加载到显存里就需要 20GB。而在训练过程中，加上梯度、优化器状态和激活值，实际需要的显存会暴涨到 **120GB 以上**，远远超过了单张 A100 (80GB) 或 H100 的极限。

为了打破这个物理硬件壁垒，微软（Microsoft）在 2020 年推出了专门针对超大规模深度学习的分布式加速库 —— **DeepSpeed**。时至今日，它依然是整个大模型工业界的统治级基础设施。

---

## 一、 DeepSpeed 的核心封神科技：ZeRO 技术

DeepSpeed 之所以能名震江湖，完全是因为它提出了一套惊天动地的显存优化方案 —— **ZeRO（Zero Redundancy Optimizer，零冗余优化器）**。

在传统的 PyTorch DDP（数据并行）中，每张显卡都会**完完整整地复制一份**模型的参数、梯度和优化器状态。这就好比 8 个工人协作盖楼，但每个人手里都拿着一本一模一样的、厚达 1000 页的施工图纸，极其浪费空间。

ZeRO 的核心思想就是：**把这本厚图纸拆散，每人只拿 125 页。当工人 A 需要用到第 200 页的内容时，再临时找工人 B 借来看一眼，用完立刻还回去（释放显存）。**

ZeRO 划分为三个渐进的阶段（Stages）：

### 1. ZeRO-Stage 1：优化器状态切片（Optimizer State Partitioning）

在 AdamW 优化器中，为了跟踪每个参数的动量，优化器占据了训练中 **75% 的静态显存**。Stage 1 将优化器状态均匀切碎分摊到每张卡上。

* **效果**：显存占用暴减到原来的四分之一左右，**速度几乎没有任何损失**。工业界最常用、最稳健的阶段。

### 2. ZeRO-Stage 2：梯度切片（Gradient Partitioning）

在 Stage 1 的基础上，把反向传播算出来的“梯度（Gradients）”也切碎分摊。每张卡只负责更新自己手里的那一小块梯度。

* **效果**：显存进一步暴减，允许你塞下更大的 Batch Size。

### 3. ZeRO-Stage 3：模型参数切片（Parameter Partitioning）

终极压榨。连最核心的“模型参数（Parameters）”也拆散分摊到所有卡上。在前向传播和反向传播时，各卡之间通过极速的网络通信（All-Gather）动态拼接出当前层所需的参数，计算完后立刻原地销毁。

* **效果**：**理论上可以训练无限大的模型**。只要你的显卡集群数量足够多，千亿参数模型也能硬生生塞进去。

---

## 二、 穷人的劳斯莱斯：ZeRO-Offload（显存不够，内存来凑）

如果你手里没有成百上千张 A100 显卡，只有 1 张或者几张普通的消费级显卡（如 RTX 3090 / 4090），你该怎么微调大模型？

DeepSpeed 推出了 **ZeRO-Offload** 魔法。

它的逻辑非常残暴：GPU 显存太贵太小，但服务器的 CPU 内存（RAM）和固态硬盘（NVMe SSD）通常很大且很便宜。
DeepSpeed 允许你在训练时，**把占据显存大头的“优化器状态”和“梯度”直接卸载（Offload）到 CPU 内存甚至是 SSD 硬盘中**，让 CPU 帮忙计算优化器更新，而 GPU 只专注于计算高密度的前向和反向传播。

* **震撼的效果**：借助 ZeRO-Offload，单张 RTX 3090（24G 显存）原本只能跑 10 亿参数的模型，现在能奇迹般地跑起 **130 亿参数（13B）** 的模型，直接让算力平民化。

---

## 三、 DeepSpeed 的高级生态模块

随着迭代，DeepSpeed 已经不仅仅局限于训练（Training），它延伸出了三大核心支柱：

1. **DeepSpeed-Training**：即上述的 ZeRO 体系，专注于百亿、千亿级大模型的**极致高效率预训练与微调**。
2. **DeepSpeed-Inference**：大模型训练完后要部署。该模块针对大模型推理进行了极致的算子融合（Transformer Kernel Fusion），极大地提升了文本生成的 Token 吞吐量，降低延迟。
3. **DeepSpeed-Chat**：一键式、流水线式的 RLHF（基于人类反馈的强化学习）训练系统。当年 ChatGPT 刚火时，它是市面上极少数能搞定完整的 PPO 三阶段强化学习训练的开源框架。

---

## 📝 总结：DeepSpeed 的历史地位

在目前的 AI 工业界，我们可以得出这样一个共识：

* **原生的 PyTorch DDP** 是公路上的**轿车**，适合运送常规大小的模型。
* **DeepSpeed** 则是重型**矿用卡车**。当你的模型体积庞大到普通车辆根本装不下、走不动时，DeepSpeed 会把模型大卸八块分配装车（ZeRO 1/2/3），甚至拉上周边的工具车协同作业（CPU Offload），直到把这个庞然大物平稳、高效地运送到终点。


---

## 一、 ZeRO 技术

## 1. 混合精度训练与显存占用

**模型参数规模**：总参数量为 $\Psi$（例如 7B 模型 $\Psi=7\times10^9$）。

在**混合精度 + Adam** 训练中，每张 GPU 需要维护的“模型状态”如下：

| 状态 | 精度 | 每个参数的字节数 | 总字节数 |
|------|------|-----------------|----------|
| 模型参数（前向用） | FP16 | 2 | $2\Psi$ |
| 梯度 | FP16 | 2 | $2\Psi$ |
| 优化器状态：FP32 主参数 | FP32 | 4 | $4\Psi$ |
| 优化器状态：一阶动量 $m$ | FP32 | 4 | $4\Psi$ |
| 优化器状态：二阶动量 $v$ | FP32 | 4 | $4\Psi$ |
| **合计** | | | **$16\Psi$** |

即单张 GPU 存储这些基础状态就需要 $16\Psi$ 字节。当 $\Psi$ 较大时（如 100B 模型需 1.6 TB），单卡远不能容纳。传统数据并行（DP）在 $N$ 张卡上每张都存完整 $16\Psi$，冗余 $N$ 倍。ZeRO 的目标是把这 $16\Psi$ 沿数据并行维度切分到 $N$ 张卡上。

---

## 2. 梯度计算的基础数学

考虑一个 $L$ 层的神经网络，参数 $\theta = \{W_1,\dots,W_L\}$。给定输入样本 $x$，前向传播为：

$$
a^{(0)} = x,\qquad 
z^{(l)} = a^{(l-1)} W_l,\qquad 
a^{(l)} = \sigma_l(z^{(l)}),\; l=1,\dots,L
$$

最终输出 $\hat{y}=a^{(L)}$，单样本损失为 $\ell(\hat{y}, y)$。

在数据并行下，全局 batch $\mathcal{B}$ 被均分到 $N$ 张 GPU 上，GPU $k$ 的局部 batch 为 $\mathcal{B}_k$，$|\mathcal{B}_k| = B/N$。全局损失是各局部损失的平均：

$$
\mathcal{L}(\theta) = \frac{1}{N}\sum_{k=0}^{N-1} L_k(\theta),\quad 
L_k(\theta) = \frac{1}{|\mathcal{B}_k|}\sum_{i\in\mathcal{B}_k} \ell(f(x^{(i)};\theta), y^{(i)})
$$

反向传播时，GPU $k$ 对每一层 $l$ 计算出**局部梯度**（基于自身数据）：

$$
G_k^{(l)} = \nabla_{W_l} L_k(\theta) \quad \text{(FP16)}
$$

该梯度是对完整权重 $W_l$ 的导数，大小为 $|W_l|$。由于各 GPU 数据不同，$G_k^{(l)}$ 互异。**全局真实的梯度**为局部梯度的平均：

$$
\nabla_{W_l}\mathcal{L}(\theta) = \frac{1}{N}\sum_{k=0}^{N-1} G_k^{(l)}
$$

分布式训练的关键就是通过通信高效地计算这一全局平均梯度，并用它更新参数。

---

## 3. 通信原语的数学定义

设 $X_0,\dots,X_{N-1}$ 为 $N$ 个 GPU 上的张量（大小均为 $D$）。

- **All-Reduce**：计算全局和并广播给所有进程。
  $$
  \forall k,\ Y_k = \sum_{j=0}^{N-1} X_j
  $$
  通信量：每卡发送和接收约 $2D$（Ring 算法）。若直接输出平均值，只需内部乘以 $1/N$。

- **Reduce-Scatter**：先求和，再将结果按维度 $D$ 切为 $N$ 块，每块分给不同的 GPU。
  设 $X_j[r]$ 为 GPU $j$ 上张量的第 $r$ 个分片（$0\le r<N$），则
  $$
  \forall k,\ Y_k = \sum_{j=0}^{N-1} X_j[k]
  $$
  即 GPU $k$ 只得到第 $k$ 个分片的全局和。通信量仍为 $2D$，但每卡最终持有 $D/N$ 的数据。

- **All-Gather**：收集各 GPU 上的分片并拼接，再分发给所有 GPU。
  $$
  \forall k,\ Y_k = [X_0, X_1, \dots, X_{N-1}]
  $$
  通信量约为 $D$（每卡发送自己的 $D/N$ 分片，接收其他 $N-1$ 块）。

这些原语构成了 ZeRO 通信的基础。

---

## 4. ZeRO Stage 1：优化器状态分片 ($P_{os}$)

### 4.1 分片策略与单卡显存
- **参数**：完整 FP16 参数，每卡均持有 $2\Psi$。
- **梯度**：完整 FP16 梯度，每卡 $2\Psi$。
- **优化器状态**（FP32 主参数、$m$、$v$）：均分为 $N$ 片，GPU $k$ 只存 $\frac{12\Psi}{N}$ 字节。

单卡模型状态显存：
$$
M_1 = 2\Psi + 2\Psi + \frac{12\Psi}{N} = 4\Psi + \frac{12\Psi}{N}
$$
当 $N=8$，约 $5.5\Psi$（相比 $16\Psi$ 节省约 3 倍）。

### 4.2 前向与反向（局部梯度计算）
GPU $k$ 用完整 FP16 参数 $\theta^{\text{fp16}}$ 进行前向传播，得到 $L_k$，然后反向计算出**完整尺寸的局部梯度** $G_k^{(l)}$（对所有 $l$）。此步骤的计算量与传统 DP 完全相同。  
Stage 1 和 Stage 2 的激活值会被完整保留

### 4.3 梯度同步：All-Reduce
对所有层 $l$ 的局部梯度执行 All-Reduce，求全局平均梯度：
$$
\bar{G}^{(l)} = \frac{1}{N}\sum_{j=0}^{N-1} G_j^{(l)} \quad \text{(FP16)}
$$
All-Reduce 的通信量为 $2\Psi$（每卡发送 $2\Psi$ 数据）。完成后，每张 GPU 都拥有**完整且一致**的全局平均梯度。

### 4.4 优化器更新（分片更新 + All-Gather 参数）
梯度已全局化，但优化器状态是分片的。GPU $k$ 只负责更新**自己拥有的那一部分参数分片**（记为分片 $s$）。对于分片 $s$ 中的每个参数：

1. 将 FP16 梯度 $\bar{G}_s$ 转为 FP32，记为 $g_s$。
2. 更新一阶、二阶矩（Adam）：
   $$
   m_s \leftarrow \beta_1 m_s + (1-\beta_1) g_s
   $$
   $$
   v_s \leftarrow \beta_2 v_s + (1-\beta_2) g_s^2
   $$
3. 偏差校正与参数更新：
   $$
   \hat{m}_s = \frac{m_s}{1-\beta_1^t},\quad \hat{v}_s = \frac{v_s}{1-\beta_2^t}
   $$
   $$
   \theta_s^{\text{fp32}} \leftarrow \theta_s^{\text{fp32}} - \eta \frac{\hat{m}_s}{\sqrt{\hat{v}_s} + \epsilon}
   $$
4. 将 $\theta_s^{\text{fp32}}$ 转为 FP16，得到新的 $\theta_s^{\text{fp16}}$。

此时，每张 GPU 只有部分分片的 FP16 参数被更新，其余仍是旧值。因此需要一次 **All-Gather**：每张卡广播自己更新的分片，拼出完整的 $\theta^{\text{fp16}}$。All-Gather 通信量为 $\Psi$。

**总通信量**：All-Reduce 梯度 $2\Psi$ + All-Gather 参数 $\Psi$ = $3\Psi$（但梯度同步的核心路径是 $2\Psi$，All-Gather 可与下一轮计算重叠）。

---

## 5. ZeRO Stage 2：梯度分片 ($P_{os}+P_g$)

### 5.1 分片策略与单卡显存
- **参数**：完整 FP16 参数，$2\Psi$。
- **梯度**：分片存储，GPU $k$ 仅保留 $\frac{2\Psi}{N}$。
- **优化器状态**：分片存储，$\frac{12\Psi}{N}$。

单卡显存：
$$
M_2 = 2\Psi + \frac{2\Psi}{N} + \frac{12\Psi}{N}
$$
$N=8$ 时约 $3.75\Psi$。

### 5.2 前向与反向（仍计算完整局部梯度）
GPU $k$ 依然用完整参数进行前向，得到 $L_k$，并反向算出**完整的局部梯度** $G_k^{(l)}$（FP16，大小 $\Psi$）。计算量与 Stage 1 一致。   
Stage 1 和 Stage 2 的激活值会被完整保留  

从表面看，Stage 2 的梯度都被切碎了，自然会让人怀疑它的前向和反向计算是不是也“只算了一部分”。

但是：**Stage 2 的前向和反向传播，在计算上与 Stage 1 完全一样，没有任何区别。** 两者都是**为整个模型的所有参数计算完整的局部梯度**。

区别仅仅在于计算完成后的**通信和存储步骤**。下面我详细解释为什么。

---

#### 5.2.1. 为什么计算必须一样？—— 参数完整性决定了一切

Stage 1 和 Stage 2 有一个共同的前提：**每张 GPU 都持有完整的 FP16 模型参数**。

- 既然拥有完整的参数，那么前向传播就必然要从第一层算到最后一层，为输入数据计算出完整的激活序列。
- 既然算出了完整的激活序列，那么反向传播就必然要从最后一层算回第一层，为**每一个参数**计算出它对应的局部梯度。

链式法则不认“分片”，它只认参数和激活值是否存在。参数都在，反向传播就会自动求出所有权重的梯度，这是无法跳过的。

**所以，从计算图的角度看，Stage 2 的每张 GPU 和 Stage 1 的每张 GPU 干了一模一样的活儿：都用自己的数据，为整个模型的所有参数算出了一份完整的局部梯度（大小仍然为 $\Psi$）。**

---

#### 5.2.2. 区别只在计算完成之后：通信和存储

计算完成时，每张 GPU 的显存里都有一份完整的 FP16 局部梯度。接下来的处理才是分水岭：

- **Stage 1**：调用 **All-Reduce**。所有卡交换完整的梯度并求和，最后**每张卡都得到一份完整的全局平均梯度**（大小 $\Psi$）。这份完整梯度被保留下来，用于更新优化器状态，并随后广播参数。
- **Stage 2**：调用 **Reduce-Scatter**。所有卡交换梯度并求和，但在求和的同时，结果被切碎并分散——**每张卡只收到自己负责的那一小块（大小 $\Psi/N$）的全局平均梯度，其余部分在通信过程中直接被丢弃**。

因此，Stage 2 节省显存的秘诀不是“少算了”，而是“算完了，合并了，然后把不需要的部分立刻扔了”。


### 5.3 梯度同步与分片：Reduce-Scatter
不再使用 All-Reduce，而是用 **Reduce-Scatter** 直接产生分片后的全局平均梯度。
将 $G_k^{(l)}$ 沿参数维度均分为 $N$ 片，记 $G_k^{(l)}[r]$ 为 GPU $k$ 上第 $r$ 个分片。Reduce-Scatter 操作输出：
$$
\bar{G}_k^{(l)} = \frac{1}{N}\sum_{j=0}^{N-1} G_j^{(l)}[k] \quad \text{(FP16)}
$$
GPU $k$ 得到**只属于自己分片 $k$ 的那部分全局平均梯度**，其余部分在通信完成后立即丢弃。
Reduce-Scatter 的通信量仍是 $2\Psi$（与 All-Reduce 相同），但每卡梯度存储降为 $2\Psi/N$。

### 5.4 优化器更新 + All-Gather 参数
每张 GPU $k$ 用自己手中的梯度分片 $\bar{G}_k$（对应各层的一部分参数），更新对应的优化器状态和 FP32 主参数（Adam 公式同 Stage 1），然后生成新的 FP16 参数分片 $\theta_k^{\text{fp16}}$。

随后执行 **All-Gather**：收集各 GPU 新参数分片，拼出完整 $\theta^{\text{fp16}}$ 分发所有卡。All-Gather 通信量 $\Psi$。

**总通信量**：Reduce-Scatter $2\Psi$ + All-Gather $\Psi$ = $3\Psi$，与 Stage 1 持平，但梯度冗余被消除。

---

## 6. ZeRO Stage 3：参数分片 ($P_{os}+P_g+P_p$)

### 6.1 分片策略与单卡显存
- 所有模型状态全部切分：参数、梯度、优化器状态均只存 $1/N$。
单卡模型状态显存：
$$
M_3 = \frac{2\Psi}{N} + \frac{2\Psi}{N} + \frac{12\Psi}{N} = \frac{16\Psi}{N}
$$
$N=8$ 时仅 $2\Psi$，实现线性降低。**训练过程中没有任何一张卡拥有完整模型**。

### 6.2 前向传播（逐层 All-Gather 参数）
对于第 $l$ 层，每张 GPU 只持有参数分片 $\theta_{k,l}^{\text{fp16}}$。

1. **All-Gather**：收集所有分片，拼出该层完整权重 $W_l^{\text{fp16}}$。
   $$
   W_l^{\text{fp16}} \leftarrow \text{All-Gather}(\theta_{0,l}^{\text{fp16}},\dots,\theta_{N-1,l}^{\text{fp16}})
   $$
   通信量 $\Psi_l$（该层参数量）。
2. **前向计算**：GPU $k$ 利用自己本地的输入 $a_k^{(l-1)}$ 计算：
   $$
   z_k^{(l)} = a_k^{(l-1)} W_l,\quad a_k^{(l)} = \sigma_l(z_k^{(l)})
   $$
3. **丢弃非本地分片**：立即释放不属于自己的 $W_l$ 部分，仅保留自己的 $\theta_{k,l}^{\text{fp16}}$。
全模型前向总通信量：$C_{\text{fwd}}=\sum_l \Psi_l = \Psi$。

前向传播算出的激活值，在 Stage 3 的实际使用中，**不会全部保留**，而是几乎全部扔掉，等反向传播需要时再临时重算。

**如果不加干涉**：前向传播会正常保留所有层的 $A_1, A_2, \dots, A_L$，但这会消耗巨大的显存，让 Stage 3 省下的显存全白费了。

**实际做法（分区激活检查点）**：
- 前向时：每算完一层，只保留该层的**输入**（即上一层的输出）作为“检查点”，中间结果全部丢弃。
- 反向时：需要哪一层的完整激活值，就从检查点出发，临时重算这一层的前向，用完立刻丢弃。

### 6.3 反向传播（逐层 All-Gather + Reduce-Scatter）
反向从最后一层开始，对于第 $l$ 层：

1. **All-Gather 参数**：再次收集完整 $W_l^{\text{fp16}}$（通信量 $\Psi_l$）。
2. **重算激活值**（若开启激活检查点）：从保存的 $a_k^{(l-1)}$ 重新计算 $z_k^{(l)}, a_k^{(l)}$。
3. **局部梯度计算**：
   接收上游梯度 $\delta_k^{(l)}$（损失对 $a_k^{(l)}$ 的梯度），计算对输入的梯度：
   $$
   \delta_k^{(l-1)} = \delta_k^{(l)} (W_l)^\mathsf{T} \odot \sigma'_l(z_k^{(l)})
   $$
   并计算对**完整权重**的局部梯度：
   $$
   G_k^{(l)} = \nabla_{W_l} L_k = \frac{1}{|\mathcal{B}_k|}\sum_{i\in\mathcal{B}_k} (a_{k,i}^{(l-1)})^\mathsf{T} \delta_{k,i}^{(l)} \quad \text{(FP16, 尺寸 }\Psi_l\text{)}
   $$
4. **Reduce-Scatter 梯度分片**：
   将 $G_k^{(l)}$ 分为 $N$ 片，执行 Reduce-Scatter：
   $$
   \bar{G}_k^{(l)} = \frac{1}{N}\sum_{j=0}^{N-1} G_j^{(l)}[k]
   $$
   GPU $k$ 仅保留自己负责的第 $k$ 个梯度分片，通信量 $\Psi_l$。
5. **丢弃完整参数**，释放非分片梯度。

反向总通信量：$C_{\text{bwd}} = \sum_l (\Psi_l + \Psi_l) = 2\Psi$。

### 6.4 优化器更新（无需通信）
反向结束后，GPU $k$ 持有：
- 参数分片 $\theta_k^{\text{fp16}}$
- 对应梯度分片 $\bar{G}_k$
- 优化器状态 $m_k, v_k, \theta_k^{\text{fp32}}$

直接本地执行 Adam 更新（公式同前），得到新 $\theta_k^{\text{fp16}}$。**无需任何 All-Gather**，因为参数本就是分片的，下一轮前向需要的分片已在本地，并在 All-Gather 时提供给其他卡。

**总通信量**：前向 $\Psi$ + 反向 $2\Psi$ = **$3\Psi$**。相比 Stage 2 的总通信 $3\Psi$（Reduce-Scatter $2\Psi$ + All-Gather $\Psi$），Stage 3 将原本的 All-Gather 参数步骤换成了前向的额外一次 All-Gather，本质上通信量并未显著增加，仅重排了通信顺序，且可通过计算重叠隐藏。

---

## 7. 梯度更新的统一视角

无论哪个 ZeRO 阶段，数学上参数更新的最终结果都与单卡大 batch 训练等价。其抽象流程为：

1. **局部梯度计算**：$\forall k,\ G_k = \nabla_\theta L_k(\theta)$，每张卡产生基于自己数据的完整梯度。
2. **全局平均**：通过通信计算 $\bar{G} = \frac{1}{N}\sum_k G_k$。
3. **分片更新**：每张卡取 $\bar{G}$ 中自己负责的分片 $\bar{G}_k$，执行 Adam 更新：
   $$
   m_k \leftarrow \beta_1 m_k + (1-\beta_1)\bar{G}_k,\quad
   v_k \leftarrow \beta_2 v_k + (1-\beta_2)\bar{G}_k^2
   $$
   $$
   \theta_k^{\text{fp32}} \leftarrow \theta_k^{\text{fp32}} - \eta\frac{m_k/(1-\beta_1^t)}{\sqrt{v_k/(1-\beta_2^t)}+\epsilon}
   $$
   并将 $\theta_k^{\text{fp32}}$ 转回 FP16。
4. **参数重组**：若参数未分片（Stage 1/2），需要通过 All-Gather 收集各卡更新后的参数分片恢复完整模型；若参数已分片（Stage 3），则直接使用本地分片，下一轮前向时按需 All-Gather。

通信的差异仅在于**如何实现步骤 2 和 4**：
- Stage 1：步骤 2 用 All-Reduce（产生完整梯度），步骤 4 用 All-Gather 参数。
- Stage 2：步骤 2 用 Reduce-Scatter（直接产生梯度分片），步骤 4 用 All-Gather 参数。
- Stage 3：步骤 2 用 Reduce-Scatter（梯度分片），步骤 4 无全局参数重组，但步骤 1 和 2 的每次计算都需要先 All-Gather 参数（前向和反向各一次）。

所有这些操作保证了 $\theta^{\text{fp32}}$ 的更新轨迹与单卡大 batch 训练完全一致。

---

## 8. 三阶段对比总结

| 阶段 | 分片内容 | 单卡模型状态显存 (N卡) | 梯度同步原语 | 参数重组原语 | 总通信量 (相对 $\Psi$) |
|------|---------|------------------------|--------------|--------------|------------------------|
| **1** | 仅优化器状态 | $2\Psi+2\Psi+12\Psi/N$ | All-Reduce ($2\Psi$) | All-Gather ($\Psi$) | $3\Psi$ |
| **2** | + 梯度 | $2\Psi+2\Psi/N+12\Psi/N$ | Reduce-Scatter ($2\Psi$) | All-Gather ($\Psi$) | $3\Psi$ |
| **3** | + 参数 | $16\Psi/N$ | Reduce-Scatter ($2\Psi$)（反向） | 前向 All-Gather ($\Psi$) + 反向 All-Gather ($\Psi$) | $3\Psi$ |

所有阶段计算量保持不变，通信量也基本相当（$3\Psi$ 量级），ZeRO 用 **通信换显存** 的哲学实现了从百亿到万亿参数模型的训练可能。